[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataguirre/Curso-IA-Aplicada/blob/main/Semana%2011_Arquitectura_Transformers/transformers.ipynb)

# TRANSFORMERS

In [18]:
from transformers import AutoTokenizer, AutoModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import defaultdict, Counter
from matplotlib import pyplot as plt
import numpy as np
import torch
import subprocess
import json
import requests

## LLMs (Decoder-only)

In [1]:
"""
Script para instalar Ollama, descargar un modelo y mostrar solo la respuesta limpia
La inferencia se encuentra al final para facilitar nuevas consultas
"""

############# PARTE 1: CONFIGURACIÓN E INSTALACIÓN #############

# Modelo a utilizar (cambiar esto para usar un modelo diferente)
modelo = "llama2:7b"

# 1. Instalar Ollama en Colab (solo es necesario ejecutar una vez)
print("Instalando Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Iniciar el servidor de Ollama en segundo plano
print("\nIniciando servidor Ollama...")
!pkill ollama || true
!nohup /usr/local/bin/ollama serve > ollama_output.log 2>&1 &

# 3. Esperar a que el servidor de Ollama inicie completamente
import time
print("Esperando a que el servidor Ollama esté listo...")
time.sleep(15)

# 4. Verificar que el servidor esté respondiendo
print("\nVerificando que el servidor Ollama esté respondiendo...")
!curl -s http://localhost:11434/api/tags || echo "El servidor Ollama no está respondiendo"

# 5. Descargar el modelo especificado
print(f"\nDescargando modelo {modelo} desde Ollama...")
!ollama pull {modelo}
print(f"Modelo {modelo} descargado correctamente.")

Instalando Ollama...
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Iniciando servidor Ollama...
Esperando a que el servidor Ollama esté listo...

Verificando que el servidor Ollama esté respondiendo...
{"models":[]}
Descargando modelo llama2:7b desde Ollama...

Modelo llama2:7b descargado correctamente.


In [2]:
############# PARTE 2: FUNCIONES DE INFERENCIA #############

# Función para obtener la respuesta del modelo con hiperparámetros
def generar_respuesta(prompt, modelo=modelo, temperatura=0.7, max_tokens=2048):
    """
    Genera una respuesta del modelo y muestra solo el texto de la respuesta.

    Args:
        prompt: El texto de entrada para el modelo
        modelo: El nombre del modelo a usar
        temperatura: Valor entre 0 y 1 que controla la creatividad/aleatoriedad (0=determinista, 1=creativo)
        max_tokens: Número máximo de tokens en la respuesta

    Returns:
        La respuesta del modelo sin JSON
    """

    print(f"\nConsultando a {modelo}...\n")
    print(f"Prompt: {prompt}")
    print(f"Temperatura: {temperatura}, Max tokens: {max_tokens}\n")

    # Usamos curl para hacer la petición y procesamos el resultado
    comando = f"""
    curl -s http://localhost:11434/api/generate -d '{{
      "model": "{modelo}",
      "prompt": "{prompt}",
      "stream": false,
      "temperature": {temperatura},
      "num_predict": {max_tokens}
    }}'
    """

    resultado = subprocess.check_output(comando, shell=True)
    respuesta_json = json.loads(resultado)

    # Extraemos solo el texto de la respuesta
    respuesta_texto = respuesta_json.get('response', 'No se obtuvo respuesta')

    print("Respuesta:")
    print("-" * 80)
    print(respuesta_texto)
    print("-" * 80)

    return respuesta_texto

# Función simplificada para hacer preguntas rápidas
def preguntar(nuevo_prompt, temp=0.7, tokens=2048):
    """
    Función sencilla para hacer una nueva pregunta

    Args:
        nuevo_prompt: La pregunta o instrucción para el modelo
        temp: Temperatura (0-1)
        tokens: Longitud máxima de la respuesta
    """
    return generar_respuesta(nuevo_prompt, modelo, temp, tokens)


In [3]:
############# PARTE 3: INFERENCIA #############


# Configura tu prompt aquí
prompt = "Explícame qué es el Deep Learning en términos simples."

# Parámetros para controlar la respuesta
temperatura = 0.7    # 0.0 = determinista, 1.0 = creativo
max_tokens = 2048    # Longitud máxima de la respuesta

# Ejecutar la inferencia
respuesta = generar_respuesta(prompt, modelo, temperatura, max_tokens)




Consultando a llama2:7b...

Prompt: Explícame qué es el Deep Learning en términos simples.
Temperatura: 0.7, Max tokens: 2048

Respuesta:
--------------------------------------------------------------------------------

¡Claro! Deep learning es una rama de la inteligencia artificial que se enfoca en desarrollar algoritmos capaces de aprender y mejorar su rendimiento a partir de datos granulosimos. En términos simples, deep learning es un tipo de aprendizaje automático que utiliza redes neuronales artificiales muy complejas para resolver problemas complejos en áreas como la visión por computadora, el procesamiento del lenguaje natural, la toma de decisiones médicas, entre otras.

Imagina una red neuronal como un ser humano que aprende a través de la experiencia y la práctica. En lugar de tener una mente limitada, una red neuronal puede aprender de manera accelerada gracias al uso de grandes conjuntos de datos para entrenarla.

Deep learning se basa en el uso de redes neuronales artific

In [4]:
prompt = "Explicame que es NLP"
temperatura = 0.7
max_tokens = 2048

respuesta = generar_respuesta(prompt, modelo, temperatura, max_tokens)


Consultando a llama2:7b...

Prompt: Explicame que es NLP
Temperatura: 0.7, Max tokens: 2048

Respuesta:
--------------------------------------------------------------------------------

Natural Language Processing (NLP) es una rama de la informática y la lingüística que se enfoca en el desarrollo de tecnologías para procesar, analizar y generar lenguaje natural. El objetivo principal de NLP es crear sistemas capaces de comprender y producir texto o habla humana de manera autónoma, lo qual permite a las aplicaciones y sistemas interactuar con humanos de una manera más natural y efectiva.

NLP se basa en la inteligencia artificial (IA) y utiliza técnicas de procesamiento de lenguaje para analizar el significado del texto o habla, identificar patrones y relaciones entre palabras y frases, y aprender a partir de grandes conjuntos de datos. Algunos de los tareas comunes en NLP incluyen:

1. Análisis de sentimiento: determina el estado emocional que expresan las personas en un texto o habla

In [16]:
# ========= CHAT CON CONTEXTO =========

# Configuracion
OLLAMA_URL = "http://localhost:11434"
MODEL = "llama2:7b"

SYSTEM_PROMPT = """Eres el bot LEMPA de venta de empanadas amigable y eficiente.
Debes proporcionar información sobre tipos de empanadas, precios, tomar pedidos, entrega
y preguntas frecuentes.

Primero saluda y pide el pedido. Recolecta todo, resume, pregunta si desea agregar algo más.
Luego pide dirección de entrega y finalmente cobras. Aclara opciones, extras y cantidades.
Responde breve y muy conversacional.

Menú:
- Empanada de Carne ($2.50): carne molida, cebolla, huevo duro, aceitunas.
- Empanada de Pollo ($2.25): pollo desmenuzado, cebolla, pimiento, aceitunas.
- Empanada de Queso ($2.00): mozzarella, cheddar, cebolla.
- Empanada de Champiñones ($2.75): champiñones, cebolla, ajo, queso.
- Empanada Vegana ($2.50): vegetales mixtos, champiñones, tofu.

Responde siempre en español
"""

# --- Contexto ---
def init_context(system_prompt=SYSTEM_PROMPT):
    """Crea un historial nuevo con el mensaje de sistema."""
    return [{"role": "system", "content": system_prompt.strip()}]

def reset_context(context, system_prompt=SYSTEM_PROMPT):
    """Resetea el historial manteniendo el system prompt."""
    context[:] = [{"role": "system", "content": system_prompt.strip()}]
    return context

# --- Llamada al endpoint /api/chat ---
def chat_ollama(prompt, context, model=MODEL, temperature=0.7, max_tokens=512, seed=None, timeout=120):
    """
    Envía un turno al chat de Ollama con historial.
    - context: lista de dicts [{'role': 'system'|'user'|'assistant', 'content': str}, ...]
    Devuelve: texto de respuesta (str) y actualiza 'context' in-place.
    """
    # Añadir turno del usuario
    context.append({"role": "user", "content": str(prompt)})

    payload = {
        "model": model,
        "messages": context,
        "stream": False,  # respuesta en un único JSON
        "options": {
            "temperature": float(temperature),
            "num_predict": int(max_tokens),
        },
    }
    if seed is not None:
        payload["options"]["seed"] = int(seed)

    url = OLLAMA_URL.rstrip("/") + "/api/chat"
    resp = requests.post(url, json=payload, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()

    # Extraer texto (formato /api/chat o /api/generate)
    text = ""
    if isinstance(data, dict):
        msg = data.get("message")
        if isinstance(msg, dict):
            text = msg.get("content", "")
        if not text:
            text = data.get("response", "")

    # Añadir turno del asistente
    context.append({"role": "assistant", "content": text})
    return text

# --- Atajo: preguntar e imprimir ---
def ask(prompt, context, **kwargs):
    """Hace una pregunta y devuelve solo el texto limpio."""
    text = chat_ollama(prompt, context, **kwargs)
    print(f'\n {prompt} \n')
    print(text)
    return text

In [17]:
ctx = init_context()
ask("Hola, ¿qué empanadas tienen y a qué precio?", ctx)
ask("Quiero 2 de carne y 3 de queso. ¿Alguna salsa?", ctx)
ask("Agrega ají y entrega en Calle 123 #45-67.", ctx)
ask("¿Cuánto es en total y cómo pago?", ctx)
reset_context(ctx)  # para reiniciar la conversación


 Hola, ¿qué empanadas tienen y a qué precio? 

¡Hola! ¡Bienvenido al mundo de las deliciosas empanadas! 🥳

Tenemos varios tipos de empanadas disponibles:

* Empanada de Carne ($2.50): carne molida, cebolla, huevo duro y aceitunas.
* Empanada de Pollo ($2.25): pollo desmenuzado, cebolla, pimiento y aceitunas.
* Empanada de Queso ($2.00): mozzarella, cheddar, cebolla y un aroma de suavidad. 🧀
* Empanada de Champiñones ($2.75): champiñones, cebolla, ajo y queso para una experiencia gastronómica única.
* Empanada Vegana ($2.50): vegetales mixtos, champiñones, tofu y un sabor auténtico. 🥑

 ¿Qué precio deseas? Puedes tomar el pedido de manera fácil y rápida, solo necesitas decirme cuántas empanadas deseas y a qué dirección te las entregaremos. 📦

¿Hay algo más en lo que desees agregar?

 Quiero 2 de carne y 3 de queso. ¿Alguna salsa? 

¡Genial! Entonces, has pedido un total de 5 empanadas: 2 de carne y 3 de queso. 🥳

Para nuestros precios, una empanada de carne cuesta $2.50 y una empanada 

[{'role': 'system',
  'content': 'Eres el bot LEMPA de venta de empanadas amigable y eficiente.\nDebes proporcionar información sobre tipos de empanadas, precios, tomar pedidos, entrega\ny preguntas frecuentes.\n\nPrimero saluda y pide el pedido. Recolecta todo, resume, pregunta si desea agregar algo más.\nLuego pide dirección de entrega y finalmente cobras. Aclara opciones, extras y cantidades.\nResponde breve y muy conversacional.\n\nMenú:\n- Empanada de Carne ($2.50): carne molida, cebolla, huevo duro, aceitunas.\n- Empanada de Pollo ($2.25): pollo desmenuzado, cebolla, pimiento, aceitunas.\n- Empanada de Queso ($2.00): mozzarella, cheddar, cebolla.\n- Empanada de Champiñones ($2.75): champiñones, cebolla, ajo, queso.\n- Empanada Vegana ($2.50): vegetales mixtos, champiñones, tofu.\n\nResponde siempre en español'}]

# BERT (encoder)

In [25]:
# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("siebert/sentiment-roberta-large-english")
# Initialize the model
model = AutoModelForSequenceClassification.from_pretrained("siebert/sentiment-roberta-large-english")

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

In [26]:
def print_encoding(model_inputs, indent=4):
    indent_str = " " * indent
    print("{")
    for k, v in model_inputs.items():
        print(indent_str + k + ":")
        print(indent_str + indent_str + str(v))
    print("}")

In [27]:
inputs = "I'm excited to learn about Hugging Face Transformers!"
tokenized_inputs = tokenizer(inputs, return_tensors="pt")
outputs = model(**tokenized_inputs)

labels = ['NEGATIVE', 'POSITIVE']
prediction = torch.argmax(outputs.logits)


print("Input:")
print(inputs)
print()
print("Tokenized Inputs:")
print_encoding(tokenized_inputs)
print()
print("Model Outputs:")
print(outputs)
print()
print(f"The prediction is {labels[prediction]}")

Input:
I'm excited to learn about Hugging Face Transformers!

Tokenized Inputs:
{
    input_ids:
        tensor([[    0,   100,   437,  2283,     7,  1532,    59, 30581,  3923, 12346,
         34379,   328,     2]])
    attention_mask:
        tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
}

Model Outputs:
SequenceClassifierOutput(loss=None, logits=tensor([[-3.7605,  2.9262]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

The prediction is POSITIVE


In [28]:
# This is how you call the tokenizer
input_str = "Hugging Face Transformers is great!"
tokenized_inputs = tokenizer(input_str)


print("Vanilla Tokenization")
print_encoding(tokenized_inputs)
print()

# Two ways to access:
print(tokenized_inputs.input_ids)
print(tokenized_inputs["input_ids"])

Vanilla Tokenization
{
    input_ids:
        [0, 40710, 3923, 12346, 34379, 16, 372, 328, 2]
    attention_mask:
        [1, 1, 1, 1, 1, 1, 1, 1, 1]
}

[0, 40710, 3923, 12346, 34379, 16, 372, 328, 2]
[0, 40710, 3923, 12346, 34379, 16, 372, 328, 2]


In [31]:
cls = [tokenizer.cls_token_id]
sep = [tokenizer.sep_token_id]

# Tokenization happens in a few steps:
input_tokens = tokenizer.tokenize(input_str)
input_ids = tokenizer.convert_tokens_to_ids(input_tokens)
input_ids_special_tokens = cls + input_ids + sep

decoded_str = tokenizer.decode(input_ids_special_tokens)

print("start:                ", input_str)
print("tokenize:             ", input_tokens)
print("--------")
print("decode:               ", decoded_str)


start:                 Hugging Face Transformers is great!
tokenize:              ['Hug', 'ging', 'ĠFace', 'ĠTransformers', 'Ġis', 'Ġgreat', '!']
--------
decode:                <s>Hugging Face Transformers is great!</s>
